## Aprendizaje No Supervisado

Los datos no tiene unan respuesta totalmente correcta.

El Experimento de hoy

K-Means ---> Divide los datos en K grupos diferentes y además de eso es quien encuentra esos mismo k grupos de eestudiantes con hábitos similares.

Método del codo ---> Elegir el número óptimo de clusters K

Silhuette Score ---> Medir qupe tan bien definidos están los clusters

PCA ( Principal Component Analysis ) ---> Reducir 14 dimensiones a 2 para poder visualizar los cluster

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#CLustering
from sklearn.cluster import KMeans

#silhouette Score
from sklearn.metrics import silhouette_score

#Reducción de dimensiones
from sklearn.decomposition import PCA

#Preprocesamiento
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

In [2]:
#Carga de datos y lectura

df = pd.read_csv("student_habits_performance.csv")

df = df.drop(columns=['student_id'])

df.head()


,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


In [3]:
#Guardar las etiquetas reales antes de que el modelo las vea
exam_score_real = df['exam_score'].copy()
aprueba_real = (df['exam_score'] >=60).astype(int)

#Eliminar las targets
df = df.drop(columns=['exam_score'])



In [4]:
#Tratamiento de nulos
moda_educacion = df['parental_education_level'].mode()[0]
df['parental_education_level'] = df['parental_education_level'].fillna(moda_educacion)

# Encoding de variables categóricas
#diet_quality
oe_diet = OrdinalEncoder(categories=[['Poor', 'Fair', 'Good']])
df['diet_quality'] = oe_diet.fit_transform(df[['diet_quality']]).reshape(-1)

#internet_quality
oe_net = OrdinalEncoder(categories=[['Poor', 'Average', 'Good']])
df['internet_quality'] = oe_net.fit_transform(df[['internet_quality']]).reshape(-1)

#parental_education_level
oe_edu = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master']])
df['parental_education_level'] = oe_edu.fit_transform(df[['parental_education_level']]).reshape(-1)

#--Encoding binario para variables Yes/No
df['part_time_job'] = (df['part_time_job'] =='Yes').astype(int)
df['extracurricular_participation'] = (df['extracurricular_participation'] =='Yes').astype(int)

#--One Hot Encoding para gender (nominal, sin orden)

#Crear una columna binaria por cata categoría de genero
df = pd.get_dummies(df, columns=['gender'], drop_first=True, dtype=int)

#Escalado de los datos
scaler = StandardScaler()

X_scaled = scaler.fit_transform(df)

#verficación después del escalado
print(f'Dataset listo para clustering:')
print(f'Estudiantes: {X_scaled.shape[0]} | Features: {X_scaled.shape[1]}')
print(f' Media global:  {X_scaled.mean():.6f} (debe ser aproximadamente 0)')
print(f' Std global: | {X_scaled.std():.6f} (debe ser aproximadamente 1)')

Dataset listo para clustering:
Estudiantes: 1000 | Features: 15
 Media global:  -0.000000 (debe ser aproximadamente 0)
 Std global: | 1.000000 (debe ser aproximadamente 1)


In [6]:
#Evaluar k_Means para K desde 1 hasta 10 cluster

k_valores   = range(1, 11) #valores de K a probar
inercias    = []
silhouettes = []

for k in k_valores:
    
    kmeans_k = KMeans(n_clusters=k, n_init=10, max_iter=300, random_state=42)
    
    labels_k = kmeans_k.fit_predict(X_scaled)
    
    # con k=1 es máxima (todos en un grupo); con k=N es 0 (cada punto es su propio cluster)
    inercias.append(kmeans_k.inertia_)
    
    if k >= 2:
        silhouettes.append(silhouette_score(X_scaled, labels_k))
    else:
        silhouettes.append(None)

#Tabla resumen con inercia y silhouette score de cada k
print('K | Inercia        | Silhouette')
print('-' * 35)
for k, ine, sil in zip(k_valores, inercias, silhouettes):
    sil_str = f'{sil:.4f}' if sil is not None else 'N/A'
    print(f'{k:2} | {ine:10.1f} | {sil_str}')


K | Inercia        | Silhouette
-----------------------------------
 1 |    15000.0 | N/A
 2 |    13947.4 | 0.0708
 3 |    13166.3 | 0.0734
 4 |    12380.3 | 0.0766
 5 |    11847.1 | 0.0691
 6 |    11421.6 | 0.0694
 7 |    11221.8 | 0.0606
 8 |    10952.2 | 0.0688
 9 |    10790.0 | 0.0719
10 |    10620.6 | 0.0684
